In [1]:
import pandas as pd
import numpy as np

def fix_swc_types_by_furthest_terminal(input_path, output_path=None):
    """
    读取 SWC，找到离 soma 最远的 terminal 节点，
    将 soma -> terminal 这条主干路径标记为 axon(2)，
    其余非 soma 节点全部标记为 dendrite(3)。
    """
    # 1. 加载 SWC 文件（忽略注释行，标准 7 列格式）
    names = ['id', 'type', 'x', 'y', 'z', 'r', 'parent']
    df = pd.read_csv(input_path, sep=r'\s+', comment='#', header=None, names=names)
    
    # 建立 id -> 行索引 和 id -> parent 的快速查找字典
    node_coords = {row['id']: np.array([row['x'], row['y'], row['z']]) for _, row in df.iterrows()}
    parent_dict = dict(zip(df['id'], df['parent']))
    
    # 2. 定位 Soma 节点（通常 parent == -1，或者原有 type == 1 的第一个点）
    soma_rows = df[df['parent'] == -1]
    if len(soma_rows) == 0:
        soma_rows = df[df['type'] == 1]
    
    if len(soma_rows) == 0:
        raise ValueError(f"无法在文件 {input_path} 中找到 soma (根节点 parent=-1)！")
    
    soma_id = soma_rows.iloc[0]['id']
    soma_pos = node_coords[soma_id]

    # 3. 寻找所有的末端节点 (Terminals / Tips)
    # 末端节点即：没有任何其他节点的 parent 指向它
    all_parents = set(df['parent'].values)
    all_ids = set(df['id'].values)
    terminal_ids = list(all_ids - all_parents)
    
    if not terminal_ids:
        raise ValueError(f"未在文件 {input_path} 中检测到任何 terminal 节点！")

    # 4. 计算每个 terminal 到 soma 的欧几里得距离，选出最远的一个
    terminal_dists = {
        t_id: np.linalg.norm(node_coords[t_id] - soma_pos) 
        for t_id in terminal_ids
    }
    furthest_terminal_id = max(terminal_dists, key=terminal_dists.get)

    # 5. 回溯提取最远 terminal 到 soma 的完整主干路径
    axon_path_node_ids = set()
    curr = furthest_terminal_id
    while curr != -1 and curr in parent_dict:
        axon_path_node_ids.add(curr)
        curr = parent_dict[curr]

    # 6. 重新赋予节点类型
    # 规则：
    # - soma 节点保持为 1
    # - soma 到该最远 terminal 路径上的节点设为 2 (Axon)
    # - 其余节点全部设为 3 (Basal Dendrite)
    def reassign_type(row):
        nid = row['id']
        if nid == soma_id:
            return 1
        elif nid in axon_path_node_ids:
            return 2
        else:
            return 3

    df['type'] = df.apply(reassign_type, axis=1)

    # 7. 保存更新后的 SWC 文件（如果指定了 output_path）
    if output_path:
        with open(output_path, 'w') as f:
            f.write("# Modified SWC with re-assigned axon/dendrite types\n")
            # 保证格式规范：id, type 为整数，浮点数保留合理精度
            for _, row in df.iterrows():
                f.write(f"{int(row['id'])} {int(row['type'])} {row['x']:.4f} {row['y']:.4f} {row['z']:.4f} {row['r']:.4f} {int(row['parent'])}\n")

    return df

In [2]:
# 修复单个文件
fix_swc_types_by_furthest_terminal(
    input_path=r"J:\BLA_three_types\BLA_swc_mirrow\221058_088_Sst.swc", 
    output_path=r"J:\BLA_three_types\221058_088_Sst.swc"
)

,id,type,x,y,z,r,parent
0,1,1,7141.54,5574.24,2476.06,1.222656,-1
1,2,2,7139.90,5574.52,2476.80,0.015625,1
2,3,3,7144.58,5574.20,2476.58,0.136719,1
3,4,3,7146.92,5574.06,2476.70,6.476562,3
4,5,3,7147.28,5574.48,2471.86,5.890625,4
...,...,...,...,...,...,...,...
1447,1448,3,6855.90,5353.18,2516.26,0.378906,1447
1448,1449,3,6854.38,5351.08,2517.02,0.582031,1448
1449,1450,3,6853.12,5345.08,2518.50,0.613281,1449
1450,1451,3,6852.74,5343.00,2518.86,0.437500,1450
